In [ ]:
import os

import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import pathlib
import pickle

# Seed für Reproduzierbarkeit
np.random.seed(42)
tf.random.set_seed(42)

RANDOM_STATE = 42

# Projektpfade relativ zum Notebooks-Ordner
repo_root = pathlib.Path().resolve().parent
data_dir = repo_root / "data"
models_dir = repo_root / "models"
features_dir = data_dir / "features"
splits_csv = data_dir / "splits"

models_dir.mkdir(parents=True, exist_ok=True)
features_dir.mkdir(parents=True, exist_ok=True)
splits_csv.mkdir(parents=True, exist_ok=True) if hasattr(splits_csv, 'mkdir') else None

# Feedforward Neural Network (FFNN) für Plankton-Klassifikation

Dieses Notebook trainiert und optimiert ein **Feedforward Neural Network** auf Plankton-Bildern. Wir verwenden die vorbereiteten Feature-Vektoren und testen verschiedene Netzwerk-Architekturen und Hyperparameter.

## 1. Features laden und Daten analysieren

Vorgefertigte Feature-Vektoren (Train undersampled, Val/Test vollständig) einlesen und Klassenverteilungen prüfen.

In [ ]:
# FFNN-Features (RGB, 64x64, flach) aus dem Preprocessing laden
feat_ffnn = np.load(features_dir / "ffnn_features.npz", allow_pickle=True)
X_train, X_val, X_test = feat_ffnn["X_train"], feat_ffnn["X_val"], feat_ffnn["X_test"]
y_train, y_val, y_test = feat_ffnn["y_train"], feat_ffnn["y_val"], feat_ffnn["y_test"]

# Klassenlabels aus den Splits (für Namen)
train_split = np.genfromtxt(splits_csv / "train.csv", delimiter=",", names=True, dtype=None, encoding="utf-8")
train_labels = train_split["label"]
label_encoder = LabelEncoder().fit(train_labels)
class_names = label_encoder.classes_

print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)

# Klassenverteilung prüfen
for name, labels in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    unique, counts = np.unique(labels, return_counts=True)
    dist = dict(zip(unique, counts))
    print(f"{name} Klassenverteilung: {dist}")

## 2. Datenvorverarbeitung

Die FFNN-Features wurden bereits im Preprocessing erzeugt (64x64 RGB, normalisiert auf [0,1]). Wir trainieren direkt auf den vorbereiteten Vektoren.

In [ ]:
print(f"Train/Val/Test Shapes: {X_train.shape}, {X_val.shape}, {X_test.shape}")
print(f"Wertebereich: min={X_train.min():.3f}, max={X_train.max():.3f}")

## 3. Netzwerk-Architektur und Training

Vergleiche zwei Modell-Varianten: **Basis** (L2 + Dropout) vs. **mit Batch Normalization** für natürliche Regularisierung.

Epochs auf 50 für kürzere Durchlaufzeit - kann erhöht werden

In [ ]:
# Setup
n_classes = len(np.unique(y_train))
n_features = X_train.shape[1]
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print(f"Features: {n_features}, Klassen: {n_classes}\n")

# Basis-Modell: L2 + Dropout
model_base = Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.Dropout(0.3),
    layers.Dense(n_classes, activation='softmax')
])

model_base.compile(optimizer=Adam(learning_rate=0.0005), 
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])

print("Basis-Modell:")
model_base.summary()

# Training
print("\nTraining Basis-Modell...")
history_base = model_base.fit(X_train, y_train, validation_data=(X_val, y_val),
                              epochs=50, batch_size=128, callbacks=[early_stop], verbose=0)

# Modell mit Batch Normalization
model_bn = Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(256, activation=None, kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(128, activation=None, kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(n_classes, activation='softmax')
])

model_bn.compile(optimizer=Adam(learning_rate=0.0005),
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

print("Training Modell mit Batch Normalization...")
history_bn = model_bn.fit(X_train, y_train, validation_data=(X_val, y_val),
                          epochs=50, batch_size=128, callbacks=[early_stop], verbose=0)

print("Training abgeschlossen")

## 4. Regularisierungseffekt

**Batch Normalization** stabilisiert Aktivierungen und reduziert interne Covariate Shift:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_base.history['val_accuracy'], label='Basis', linewidth=2.5)
axes[0].plot(history_bn.history['val_accuracy'], label='+ BatchNorm', linewidth=2.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('Accuracy Vergleich')
axes[0].set_ylim(0.0, 1.0)
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(history_base.history['val_loss'], label='Basis', linewidth=2.5)
axes[1].plot(history_bn.history['val_loss'], label='+ BatchNorm', linewidth=2.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Loss')
axes[1].set_title('Konvergenzverhalten')
axes[1].set_ylim(0.0, 3.0)
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Test-Evaluation

In [ ]:
# Evaluiere beide Modelle
y_pred_base = model_base.predict(X_test, verbose=0).argmax(axis=1)
y_pred_bn = model_bn.predict(X_test, verbose=0).argmax(axis=1)

acc_base = accuracy_score(y_test, y_pred_base)
acc_bn = accuracy_score(y_test, y_pred_bn)

print(f"Test-Accuracy Basis:       {acc_base:.4f}")
print(f"Test-Accuracy + BatchNorm: {acc_bn:.4f}")

# Wähle bestes Modell
best_pred = y_pred_bn if acc_bn >= acc_base else y_pred_base
best_acc = max(acc_base, acc_bn)

# Confusion Matrix
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix (Test Acc: {best_acc:.4f})')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, best_pred, target_names=[str(c) for c in class_names]))

## Modell speichern

Das beste Feedforward Neural Network Modell wird für den finalen Vergleich in Notebook 06 gespeichert.


In [ ]:
# Wähle das beste Modell
best_ffnn_model = model_bn if acc_bn >= acc_base else model_base
best_ffnn_acc = max(acc_base, acc_bn)
best_ffnn_name = "BatchNormalization" if acc_bn >= acc_base else "L2 + Dropout"

model_path = models_dir / "ffnn_best_model.h5"
best_ffnn_model.save(model_path)

# Metadaten speichern
metadata = {
    "model_type": f"Feedforward Neural Network ({best_ffnn_name})",
    "architecture": str(best_ffnn_model),
    "test_accuracy": float(best_ffnn_acc),
    "input_features": n_features,
    "n_classes": n_classes,
    "regularization": best_ffnn_name,
}

metadata_path = models_dir / "ffnn_metadata.pkl"
with open(metadata_path, "wb") as f:
    pickle.dump(metadata, f)

print(f"FFNN Modell und Metadaten gespeichert in {models_dir}")
print(f"  - Best Variant: {best_ffnn_name}")
print(f"  - Test Accuracy: {best_ffnn_acc:.4f}")
print(f"  - Input Features: {n_features}")


## 6. Fazit

**Regularisierungstechniken in Feedforward Networks:**
- **L2 Regularisierung**: Bestraft große Gewichte für glattere Entscheidungsgrenzen
- **Dropout**: Verhindert Co-Adaptierung durch stochastisches Netzwerk-Pruning  
- **Batch Normalization**: Stabilisiert Aktivierungen und beschleunigt Konvergenz
- **Early Stopping**: Bricht Training ab bei nicht-sinkendem Validierungsverlust

Die Kombination mehrerer Techniken führt zu besserer Generalisierung!